# Machine Learning for Spatial Data — C
## Graph neural networks: spatial data *is* a graph

**Companion to deck `09-spatial-machine-learning` (Module C — Graphs & GNNs).**

A PySAL spatial-weights matrix `W` (deck 05) is an **adjacency matrix**: nodes =
areas/points, edges = neighbours. That is exactly what a **graph neural network**
consumes. A GCN layer (Kipf & Welling 2017) does one step of *message passing*:
each node averages its neighbours' features, then applies a learned transform —
a **learnable, non-linear spatial lag**.

This notebook shows, step by step, how to:
1. Load the **Cora** citation network with `torch_geometric.datasets.Planetoid`
   from the local `data/Cora/` (2,708 nodes, 10,556 edges, 1,433-dim features, 7 classes).
2. Define a 2-layer **GCN** (`GCNConv`) and a feature-only **MLP** baseline that
   ignores the graph, and train both on the standard semi-supervised split.
3. Compare test accuracy: on Cora the **graph structure carries the signal** the
   node features miss, and the GCN wins by a wide margin.
4. Turn **San Diego Airbnb** into a graph — build a **KNN(8)** weights object with
   `libpysal.weights.KNN` from the listing coordinates and convert `w.neighbors`
   into a symmetric `edge_index`.
5. Standardise the listing features and the **log price** target, then train a
   `GCNReg` against an `MLPReg` on an identical 60/40 node split.
6. Score both with **R²** *and* the **Moran's I of the predictions** (`esda.moran.Moran`)
   to measure how spatially smooth each model's output is.
7. Read the flip side: here the GCN **over-smooths** — lower accuracy, far higher
   prediction autocorrelation — because the features already carry the signal.
   Use a GNN when neighbourhood structure adds information, not as a reflex.

> Datasets: `data/Cora/` — the Planetoid citation benchmark (loaded locally, no download);
> `data/airbnb/airbnb_clean.csv` — San Diego Airbnb listings with `longitude`/`latitude`,
> `price` and listing attributes (module A's data).
> Spatial-GNN case study to explore: De Sabbata & Liu (2023, *IJGIS*), Greater-London LOAC
> geodemographics, open MIT code: https://github.com/stefdesabbata/gnn-geodemographics-loac


In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
torch.manual_seed(42)
np.random.seed(42)

## 1 — Cora: when structure carries the signal

2,708 papers (nodes), 10,556 citation edges, 1,433-dim bag-of-words features, 7
classes. Standard semi-supervised split (140 train / 1,000 test). We compare a
2-layer **GCN** (uses the graph) against an **MLP** on the *same features* that
ignores the graph.

In [2]:
ds = Planetoid(root="data", name="Cora")   # loads the local data/Cora, no download
g = ds[0]
print(f"Cora: {g.num_nodes} nodes, {g.num_edges} edges, {ds.num_classes} classes")

class GCN(torch.nn.Module):
    def __init__(self, i, h, o):
        super().__init__()
        self.c1, self.c2 = GCNConv(i, h), GCNConv(h, o)
    def forward(self, x, e):
        return self.c2(F.relu(self.c1(x, e)), e)

class MLP(torch.nn.Module):
    def __init__(self, i, h, o):
        super().__init__()
        self.l1, self.l2 = torch.nn.Linear(i, h), torch.nn.Linear(h, o)
    def forward(self, x, e=None):
        return self.l2(F.relu(self.l1(x)))

def train_clf(model, use_graph):
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(200):
        model.train(); opt.zero_grad()
        out = model(g.x, g.edge_index) if use_graph else model(g.x)
        F.cross_entropy(out[g.train_mask], g.y[g.train_mask]).backward()
        opt.step()
    model.eval()
    out = model(g.x, g.edge_index) if use_graph else model(g.x)
    pred = out.argmax(1)
    return (pred[g.test_mask] == g.y[g.test_mask]).float().mean().item()

acc_mlp = train_clf(MLP(ds.num_features, 16, ds.num_classes), False)
acc_gcn = train_clf(GCN(ds.num_features, 16, ds.num_classes), True)
print(f"MLP (features only, ignores graph)  test accuracy = {acc_mlp:.3f}")
print(f"GCN (uses the citation graph)        test accuracy = {acc_gcn:.3f}")

Cora: 2708 nodes, 10556 edges, 7 classes


MLP (features only, ignores graph)  test accuracy = 0.542
GCN (uses the citation graph)        test accuracy = 0.806


The GCN beats the feature-only MLP by a wide margin: on Cora the **graph** carries
information the features alone miss. This is the classic result that launched
modern GNNs (Kipf & Welling 2017).

## 2 — Airbnb as a spatial graph: message passing = spatial smoothing

Now the course's own data. Build a **KNN(8) graph** with `libpysal` (the same
weights machinery as deck 05), turn it into `edge_index`, and predict log price
with a GCN vs an MLP on the listing features. We also measure **Moran's I of the
predictions** — how spatially smooth each model's output is.

In [3]:
from libpysal.weights import KNN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from esda.moran import Moran

df = pd.read_csv("data/airbnb/airbnb_clean.csv")
feat = ["accommodates", "bedrooms", "beds", "bathrooms", "guests_included",
        "reviews_per_month", "number_of_reviews", "host_listings_count"]
X = StandardScaler().fit_transform(df[feat].fillna(0.0).values).astype("float32")
ys = np.log1p(df["price"].values)
y = ((ys - ys.mean()) / ys.std()).astype("float32")
coords = df[["longitude", "latitude"]].values

w = KNN.from_array(coords, k=8)                    # spatial weights -> graph
rows, cols = [], []
for i, nbrs in w.neighbors.items():
    for j in nbrs:
        rows.append(i); cols.append(j)
edge_index = torch.tensor([rows, cols])
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)  # symmetric
print(f"KNN graph: {df.shape[0]} nodes, {edge_index.shape[1]} directed edges")

x = torch.tensor(X); yt = torch.tensor(y).view(-1, 1)
n = len(df); perm = torch.randperm(n)
train_mask = torch.zeros(n, dtype=torch.bool); test_mask = torch.zeros(n, dtype=torch.bool)
train_mask[perm[:int(0.6 * n)]] = True
test_mask[perm[int(0.6 * n):]] = True

KNN graph: 3173 nodes, 50768 directed edges


In [4]:
class GCNReg(torch.nn.Module):
    def __init__(self, i, h):
        super().__init__()
        self.c1, self.c2 = GCNConv(i, h), GCNConv(h, 1)
    def forward(self, x, e):
        return self.c2(F.relu(self.c1(x, e)), e)

class MLPReg(torch.nn.Module):
    def __init__(self, i, h):
        super().__init__()
        self.l1, self.l2 = torch.nn.Linear(i, h), torch.nn.Linear(h, 1)
    def forward(self, x, e=None):
        return self.l2(F.relu(self.l1(x)))

def train_reg(model):
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(400):
        model.train(); opt.zero_grad()
        F.mse_loss(model(x, edge_index)[train_mask], yt[train_mask]).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        out = model(x, edge_index).numpy().ravel()
    return r2_score(y[test_mask], out[test_mask]), Moran(out, w, permutations=0).I

r2_mlp, moran_mlp = train_reg(MLPReg(X.shape[1], 32))
r2_gcn, moran_gcn = train_reg(GCNReg(X.shape[1], 32))
print(f"actual log-price   Moran's I = {Moran(y, w, permutations=0).I:.3f}")
print(f"MLP   test R^2 = {r2_mlp:.3f}   prediction Moran's I = {moran_mlp:.3f}")
print(f"GCN   test R^2 = {r2_gcn:.3f}   prediction Moran's I = {moran_gcn:.3f}")

actual log-price   Moran's I = 0.191
MLP   test R^2 = 0.595   prediction Moran's I = 0.093
GCN   test R^2 = 0.135   prediction Moran's I = 0.911


**Lesson —** Here the GCN is *less* accurate but its predictions are
far more spatially smooth (Moran's I ≫ the actual value): message passing
**over-smooths** because the listing features already carry most of the signal —
averaging over neighbours blurs it. This is the flip side of Cora, where the
graph *was* the signal.

GNNs are not a free lunch: they **impose** spatial structure. De Sabbata & Liu
(2023, *IJGIS*) find exactly this for geodemographic classification — their graph
autoencoder gives class homogeneity similar to classic methods but **higher
spatial clustering** of the output. Use a GNN when neighbourhood structure
carries signal your node features miss; not as a reflex.

**Real spatial-GNN case study (to explore):** De Sabbata & Liu 2023 —
Greater-London LOAC geodemographics, open MIT code:
https://github.com/stefdesabbata/gnn-geodemographics-loac (reuses London/LSOA
census data). And to make a GNN *properly* spatial, inject a location encoder —
module D (PE-GNN).